In [2]:
!git clone https://github.com/Otza02/land2vec.git
%cd land2vec
!pip install -e .

fatal: destination path 'land2vec' already exists and is not an empty directory.
/content/land2vec
Obtaining file:///content/land2vec
  Preparing metadata (setup.py) ... done
  Running setup.py develop for land2vec


In [ ]:
%load_ext autoreload
%autoreload 2

In [9]:
import tqdm

import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import lr_scheduler

# from land2vec import load_data, SequenceDataset, DecoderTransformer, Config
from land2vec.dataset import load_data, SequenceDataset
from land2vec.config import Config
from land2vec.model import DecoderTransformer, train_loop, val_loop
from land2vec.tokenizer import Tokenizer

ImportError: cannot import name 'train_loop' from 'land2vec.model' (C:\Users\Admin\Desktop\unsam\proyecto\land2vec\src\land2vec\model.py)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
config = Config()

In [ ]:
print("loading data")
dataset = load_data(window=8)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
print("spliting data")
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False
)

100%|██████████| 1424457/1424457 [01:10<00:00, 20123.49it/s]


In [5]:
config

Config(block_size=8, n_embd=32, n_head=2, n_layer=2, dropout=0.1, epochs=10, batch_size=128, lr=0.001)

In [ ]:
model = DecoderTransformer(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
    dropout=config.dropout
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)
scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
for epoch in range(config.epochs):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x = x.long().to(device)
        y = y.long().to(device)
        logits, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    scheduler.step()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} | loss={avg_loss:.4f}")

Epoch 1 | loss=0.0343
Epoch 2 | loss=0.0312
Epoch 3 | loss=0.0312
Epoch 4 | loss=0.0311
Epoch 5 | loss=0.0312
Epoch 6 | loss=0.0311
Epoch 7 | loss=0.0311
Epoch 8 | loss=0.0312
Epoch 9 | loss=0.0311
Epoch 10 | loss=0.0312


In [ ]:
model = DecoderTransformer(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
    dropout=config.dropout
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=config.min_lr)

train_losses = []
val_losses = []

best_val_loss = float("inf")
patience_counter = 0

for epoch in range(config.epochs):
    train_loss = train_loop(model, train_dataset, optimizer, device)
    val_loss = val_loop(model, val_loader, device)

    scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        patience_counter = 0
    else:
        patience_counter += 1
    current_lr = scheduler.get_last_lr()[0]

    print(f"Epoch {epoch+1} \nlr={current_lr:.6f}")

    if patience_counter >= config.patience:
        print("Early stopping triggered")
        break

model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
torch.save(model.state_dict(), "first-test.pt")

In [ ]:
model = DecoderTransformer(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
).to(device)

model.load_state_dict(torch.load("models/first-test.pt"))

model.eval()

GPT(
  (token_embedding): Embedding(16, 32)
  (position_embedding): Embedding(8, 32)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=32, out_features=16, bias=True)
)

In [ ]:
model.eval()

correct = 0
total = 0

all_preds = []
all_targets = []

with torch.no_grad():
    for x, y in tqdm.tqdm(test_loader):
        x = x.long().to(device)
        y = y.long().to(device)

        logits, loss = model(x, y)

        preds = torch.argmax(logits, dim=-1)

        correct += (preds == y).sum().item()
        total += y.numel()

        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

accuracy = correct / total

print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.9953
